In [1]:
#export

#####
#####
# LLAMADA A API DE OLLAMA PARA GENERAR RESPUESTAS.
#####
#####

##### LEVEL 1 #####

import requests
import os
import json
import logging 
import logs
import re

import httpx

import mimetypes

import shlex
import subprocess
from typing import Optional, Tuple

MODELO_DEFECTO="qwen3:30b-a3b"

#MICROSOFT
MODELO_PHI4_Q8="phi4:14b-q8_0"

#QWEN
MODELO_QWEN_1_5B="qwen2.5-coder:1.5b"
MODELO_QWEN3_30B_THINK="qwen3:30b-a3b-thinking-2507-q4_K_M"
MODELO_QWEN3_30B="qwen3:30b"
MODELO_QWEN2_5VL_7B="qwen2.5vl:32b"
MODELO_QWEN3_CODER="qwen3-coder:latest"

#MISTRAL
MODELO_MISTRAL_24B_Q8="mmistral-small3.2:latest"
MODELO_DEVSTRAL="devstral:latest"

#GEMMA
MODELO_GEMMA3_27B="gemma3:27b"

#OPEN AI
MODELO_OPENAI="gpt-oss:latest"
      
ELEMENTOS_PERMITIDOS=["funcion", "clase"]


In [5]:
#export 

class OllamaException(Exception):
    """
    CLASS. 

    Represents a custom exception for errors related to calling the Ollama API.

    The `OllamaException` class inherits from the base `Exception` class and provides additional context for the error, including a custom message and any associated errors.

    The `__init__` method initializes the exception with the provided message, errors, and a default message indicating an error calling the Ollama API.
    """
    def __init__(self, message, errors="", messagio="Error calling ollama API"):

        # Call the base class constructor with the parameters it needs
        super(OllamaException, self).__init__(messagio + '-' + message, errors)

        # Now for your custom code...
        self.errors = errors


In [6]:
#export

def separa_thinking_contenido(result):
    """
    Separa el contenido de la respuesta en dos partes:
        - La parte que contiene la razonamiento (think)
        - La parte que contiene el contenido del ticket
    """
    thinking_part =""
    content_part = result
 
    start_index = result.find("<think>")
   
    if start_index > -1:
        start_index += len("<think>")
        end_index = result.find("</think>")
 
        # Extraemos la cadena de razonamiento
        thinking_part = result[start_index:end_index]
       
        # Extraemos el contenido
        content_part = result[end_index + len("</think>"):]
 
    return content_part, thinking_part


def procesa_output(response):
    """
    FUNCTION. procesa_output

    Processes the output from the Ollama API response. This function takes the response object from the Ollama API call and extracts the relevant content from the 
    JSON response. It handles different formats of the response, including the 'message' and 'response' keys, and returns a single string containing the processed output.

    Args:
        response (requests.Response): The response object from the Ollama API call.

    Returns:
        str: The processed output from the Ollama API response. 
    """
    response_str=response.content.decode('utf-8')
    response_list=response_str.split('\n')
    cadena_output=[]
    for line in response_list:
        if line.strip():
            json_obj=json.loads(line)
            
            if 'message' in json_obj and 'content' in json_obj['message']:
                content_value = json_obj['message']['content']
            elif 'response' in json_obj:
                content_value = json_obj['response']
            else:
                content_value = "error en formato json"
    
            cadena_output.append(content_value)

    cadena_final=''.join(cadena_output)
    
    return cadena_final



In [7]:
#export

def call_ollama_api_request(model, message, tipo='generate', temperature=0.5):
    """
    FUNCTION. call_ollama_api_request.

    Calls the Ollama API to generate a response based on the provided model, message, and request type.

    Args:
        model (str): The model to use for the API request.
        message (str): The message or prompt to send to the API.
        tipo (str, optional): The type of request to make, either 'generate' or another supported type as chat. Defaults to 'generate'.

    Returns:
        requests.Response: The response object from the API request.

    Raises:
        OllamaException: If the API request returns a non-200 status code.
    """

    method="call_ollama_api_request"

    url = f"http://localhost:11434/api/{tipo}"  # Cambia esta URL según tu configuración
    payload = {
        "model": model,
        "prompt": message,
        "raw" : True,
        "stream" : False,
        "options": {
            "temperature": temperature
        }
    }

    headers= {
        "Content-Type": "application/json",
    }

    logging.debug(f"{method}- url={url}, model={model}, tipo={tipo}, temperature={temperature}")
    response = requests.post(url, json=payload, headers=headers)
    if response.status_code == 200:
        return response
    else:
        raise OllamaException(f"Returned code: {response.status_code}, Response Text={response.text}")


    


In [8]:
#export

from openai import OpenAI 

api_base="http://localhost:11434/v1"
api_key="ollama"


def get_loaded_models():
    """
    Retrieves a list of all models currently loaded in Ollama.

    Returns:
        list: A list of model names.
    """
    method = "get_loaded_models"
    logging.debug(f"{method}-START")

    try:
        # Initialize the client with the API base and key
        client = OpenAI(
            base_url=api_base,
            api_key=api_key
        )

        # Make a request to get the list of loaded models
        response = client.models.list()

        # Extract the model names from the response
        models = [model.id for model in response.data]

        return models

    except Exception as e:
        logging.error(f"{method}-ERROR {str(e)}")
        raise OllamaException(f"Exception: {str(e)}")



In [9]:
#export 

def call_ollama_api(model, system_message, user_message, temperature=0.5, think=True, max_token=1000000, timeout=300.0):
    """
    FUNCTION. call_ollama_api.

    Calls the Ollama API, via python package, to generate a response based on the provided model, user message, and system_message.

    Args:
        model (str): The model to use for the API request.
        system_message (str): The message or prompt by system role to send to the API.
        user_message (str): The message or prompt by user role to send to the API.
        temperature (str, optional): temperature to ajust model imagination.

    Returns:
        requests.Response: The response object from the API request.

    Raises:
        OllamaException: If the API request returns a non-200 status code.
    """

    method="call_ollama_api"
    logging.debug(f"{method}-START with model={model}, temperature={temperature}, mensaje={user_message[0:100]}...")

    try:
        # Crear un timeout personalizado de 60 minutos (en cada fase de la petición)
        timeout = httpx.Timeout(
            connect=60.0,      # conexión: 1 minuto
            read=timeout,       # lectura de respuesta: 60 minutos
            write=60.0,        # escritura de payload: 10 minuto
            pool=None          # sin timeout en la conexión del pool
        )

        client = OpenAI(
            base_url = api_base,
            api_key = api_key,
            http_client=httpx.Client(timeout=timeout)
        )

        messages=[
            {"role":"system", "content": system_message},
            {"role": "user", "content": user_message}
        ]
                                 
        response=client.chat.completions.create(
            model=model, 
            messages=messages,
            temperature=temperature,
            extra_body={"think": think},
            max_tokens=max_token)
        
        try:
            reasoning = response.choices[0].message.reasoning
        except Exception as e:
            reasoning = None 
        content = response.choices[0].message.content 

        salida = f"<think>{reasoning}</think>{content}" if reasoning is not None  else content
        
        return salida 
    except Exception as e:
        logging.error(f"{method}-ERROR {str(e)}")
        raise OllamaException(f"Exception: {str(e)}")


In [1]:
#export 

import logging
from functools import lru_cache

from mlx_lm import load, generate


class MLXException(Exception):
    pass


@lru_cache(maxsize=3)
def _load_mlx_model(model_id: str):
    """
    Cachea el modelo para no recargarlo en cada llamada.
    """
    return load(model_id)


def call_mlx_lm_api(
    model,
    system_message,
    user_message,
    temperature=0.5,
    think=True,
    max_token=1000000,
    timeout=300.0,  # Se mantiene por compatibilidad, pero MLX local no usa timeout HTTP
):
    """
    Calls mlx-lm locally to generate a response.

    Args:
        model (str): HF/MLX model id, e.g. zaydiscold/Qwopus3.6-35B-A3B-v1-MLX-6bit
        system_message (str): System prompt.
        user_message (str): User prompt.
        temperature (float): Sampling temperature.
        think (bool): If True, asks the model to reason if the chat template/model supports it.
        max_token (int): Max tokens to generate.
        timeout (float): Ignored. Kept for API compatibility.

    Returns:
        str: Generated text.

    Raises:
        MLXException: If generation fails.
    """

    method = "call_mlx_lm_api"
    logging.debug(
        f"{method}-START with model={model}, "
        f"temperature={temperature}, mensaje={user_message[:100]}..."
    )

    try:
        mlx_model, tokenizer = _load_mlx_model(model)

        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ]

        # Algunos modelos Qwen soportan enable_thinking en apply_chat_template.
        # Si no lo soporta, caemos al template normal.
        try:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=think,
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        response = generate(
            mlx_model,
            tokenizer,
            prompt=prompt,
            max_tokens=max_token,
            temp=temperature,
            verbose=False,
        )

        return response

    except Exception as e:
        logging.error(f"{method}-ERROR {str(e)}")
        raise MLXException(f"Exception: {str(e)}")

/Users/zzddfge/.pyenv/versions/mlx-31211/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
respuesta = call_mlx_lm_api(
    model="zaydiscold/Qwopus3.6-35B-A3B-v1-MLX-4bit",
    system_message="You are a helpful assistant.",
    user_message="Explícame qué es MTP en modelos LLM.",
    temperature=0.8,
    think=True,
    max_token=4096,
)

Fetching 10 files: 100%|██████████| 10/10 [23:36<00:00, 141.66s/it]
[transformers] The tokenizer you are loading from '/Users/zzddfge/.cache/huggingface/hub/models--zaydiscold--Qwopus3.6-35B-A3B-v1-MLX-4bit/snapshots/ad881fd78e00e0f691c02cd254561769e59a93e1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
ERROR:root:call_mlx_lm_api-ERROR generate_step() got an unexpected keyword argument 'temp'


MLXException: Exception: generate_step() got an unexpected keyword argument 'temp'

In [ ]:
#export 

from mlx_vlm import load,generate
from mlx_vlm.video_generate import process_vision_info

import mlx.core as mx
from typing import Tuple

def split_thinking_and_answer(text: str) -> Tuple[str, str]:
    if not text:
        return "", ""

    text = text.strip()
    if "</think>" not in text:
        return "", text

    pattern = re.compile(r"(?:<think>\s*)?(.*?)\s*</think>\s*(.*)", re.DOTALL)
    match = pattern.fullmatch(text)

    if match:
        return match.group(1).strip(), match.group(2).strip()
        
def describe_video(
    video_path: str,
    prompt: str = "Resume brevemente el vídeo en 3-5 líneas: ¿qué se ve, quién sale y qué ocurre?. ¿Aparece una niña en el video?",
    model_id: str = "mlx-community/Qwen3.5-9B-MLX-4bit", #"mlx-community/Qwen3-VL-4B-Instruct-4bit", 
    fps: float = 1.0,
    max_pixels=(224, 224),
    max_tokens: int = 4096,
    temperature: float = 0.6,
    timeout_sec: int = 600,
) -> str:
    """ 
    The function `describe_video` takes a video file path and makes a sumarization using the model_id parameter, by default qwen2.5-vl-7b.
    """
    # Load the model and processor
    model, processor = load(model_id)
    #model, processor = load("mlx-community/SmolVLM2-2.2B-Instruct-mlx")

    messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_path,
                "max_pixels": 240 * 240,
                "fps": fps,
            },
            {"type": "text", "text": prompt},
        ],
    }]

    # Preparation for inference
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    # Convert inputs to mlx arrays
    input_ids = mx.array(inputs['input_ids'])
    pixel_values = mx.array(inputs['pixel_values_videos'])
    mask = mx.array(inputs['attention_mask'])
    video_grid_thw = mx.array(inputs['video_grid_thw'])

    kwargs = {
        "video_grid_thw": video_grid_thw,
    }

    kwargs["video"] = video_path
    kwargs["input_ids"] = input_ids
    kwargs["pixel_values"] = pixel_values
    kwargs["mask"] = mask
    #response = generate(model, processor, prompt=text, temperature=0.7, max_tokens=max_tokens, **kwargs)
    response = generate(
        model,
        processor,
        prompt=text,
        temperature=temperature,
        max_tokens=max_tokens,
        thinking_budget=2048,   # límite de tokens del bloque <think>
        input_ids=input_ids,
        pixel_values=pixel_values,
        mask=mask,
        video_grid_thw=video_grid_thw,
    )
    thinking, answer = split_thinking_and_answer(response.text)

    return answer

def describe_video2(
    video_path: str,
    prompt: str = (
        "Resume brevemente el vídeo en 3-5 líneas: qué se ve, quién sale y qué ocurre. "
        "Indica también si aparece una niña."
    ),
    model_id: str = "mlx-community/Qwen3.5-9B-MLX-4bit",
    fps: float = 1.0,
    max_pixels: tuple[int, int] = (224, 224),
    max_tokens: int = 1024,
    temperature: float = 0.2,
    thinking_budget: int | None = 256,
    enable_thinking: bool = True,
) -> str:
    """
    Describe un vídeo con un modelo VLM en MLX.

    Notas:
    - enable_thinking=True permite que el modelo use <think>...</think>.
    - thinking_budget limita cuántos tokens puede gastar dentro de ese bloque.
    - Esta versión usa una sola ruta coherente:
      1) apply_chat_template
      2) process_vision_info
      3) processor(...)
      4) generate(...) con tensores ya preparados
    """

    model, processor = load(model_id)
    
    messages = [
        {"role": "system",
         "content": [ {"type": "text", "text": system_message},],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path,
                    "max_pixels": max_pixels[0] * max_pixels[1],
                    "fps": fps,
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Construye el prompt final del chat template.
    # En Qwen3.5, thinking se controla aquí; el budget se aplica en generate.
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    input_ids = mx.array(inputs["input_ids"])
    attention_mask = mx.array(inputs["attention_mask"])
    pixel_values_videos = mx.array(inputs["pixel_values_videos"])

    gen_kwargs = {
        "input_ids": input_ids,
        "mask": attention_mask,
        "pixel_values": pixel_values_videos,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    # Algunos processors/modelos exponen video_grid_thw; si está, se pasa.
    if "video_grid_thw" in inputs:
        gen_kwargs["video_grid_thw"] = mx.array(inputs["video_grid_thw"])

    # thinking_budget está documentado en mlx-vlm reciente
    if thinking_budget is not None:
        gen_kwargs["thinking_budget"] = thinking_budget

    # IMPORTANTE:
    # No pasar prompt=text ni video=video_path aquí,
    # porque ya estamos usando los tensores preparados.
    response = generate(
        model,
        processor,
        prompt=prompt,
        **gen_kwargs,
    )

    raw_text = response.text if hasattr(response, "text") else str(response)
    _, answer = split_thinking_and_answer(raw_text)

    return answer or raw_text.strip()

In [39]:

usuario = 'Users/zzddfge'
carpeta = 'Downloads'
disco_duro = '/Volumes/Macintosh HD'

output_path=os.path.join(disco_duro, usuario, carpeta)

video = "pLv3_7Jh8rM.mp4"
video = "cm5OgIUpU-E.mp4"
video_path=os.path.join(output_path, video)
#print(summarize_video(video_path))
describe_video2(video_path)

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 140662.63it/s]
numpy reader: video_path=/Volumes/Macintosh HD/Users/zzddfge/Downloads/cm5OgIUpU-E.mp4, total_frames=132, video_fps=29.836, time=0.000s


ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).

In [ ]:
#export 

###### RESUMENES DE FICHEROS

def genera_resumen(message, length="medium", model=MODELO_DEFECTO, max_token=5000):
    method="genera_resumen"

    system_message="""
    You are an IA specialized for making summary. You are rewarded for greats summary.\n
    Follow these instructions strictly:\n
    - Extract only the most relevant information.\n
    - Keep the tone clear and concise.\n
    - Use natural and fluent language.\n
    - Do not add extra information\n
    - Before answering think about if all key points are included. Add whatever some key point is not included.\n
    - Make sure that each key point is about 5 sentences long.
    - The answer must be in spanish.\n\n
    """
    user_message=(f"Summarize the following text in a {length} way, keeping key points clear and concise :\n\n"
                  f"'''{message}'''\n\n"
                  "Summary:")

    resumen = call_ollama_api(model=model, system_message=system_message, user_message=user_message, temperature=0.3, max_token=max_token)

    return resumen

def actualiza_resumen(resumen, message, length="medium", model=MODELO_DEFECTO, max_token=5000):
    method="genera_resumen"

    system_message="""
        You are an IA specialized for updating summary. You are rewarded for greats summary. 
        You follow the instructions given strictly.\n
        You need two main input: a text, and a list of key points extracted already from the text. If you dont have both elements you should ask for them.\n
        Your mission is to update all of the key points, adding a detalied summary from the text where this point is treated.\n
    """
    
    user_message=f"""
        <system>
        Eres un asistente experto en comprensión y resumen de textos. Se te proporcionará una lista de puntos clave extraídos de un texto junto con el texto original. 
        Tu tarea es completar cada punto clave agregando un resumen detallado de su contenido, asegurándote de mantener la coherencia y precisión del mensaje original.
        </system>
        <instrucciones>
        1. Piensa paso a paso. 
        2. Antes de responder, asegurate que se incluyen todos los puntos claves de la lista proporcionada.
        3. Ubica la información relevante en el texto original.
        4. Expande cada punto clave con un resumen detallado y bien estructurado.
        5. Usa un tono claro y profesional, evitando repetir información innecesaria.
        6. Asegúrate de que cada resumen sea conciso pero informativo, sin inventar información adicional.
        </instrucciones>
        <lista puntos claves>
        **Lista de puntos clave extraídos del texto:**
        {resumen}
        </lista puntos claves>
        <texto original>
        {message}
        </texto original>
        <ejemplo salida esperada>
        ### ** Salida esperada (ejemplo de formato de respuesta):**
        **Punto 1:** [Primer punto clave]  
        **Explicación detallada:** [Resumen detallado basado en el texto original]

        **Punto 2:** [Segundo punto clave]  
        **Explicación detallada:** [Resumen detallado basado en el texto original]

        **Punto 3:** [Tercer punto clave]  
        **Explicación detallada:** [Resumen detallado basado en el texto original]

        Etc. Asegurate de incluir todos los puntos claves.
        </ejemplo salida esperada>
        <idioma>
        1. Identifica el lenguaje del texto original.
        2. Haz el resumen en el lenguaje del texto original.
        </idioma>
    """

    resumen = call_ollama_api(model=model, system_message=system_message, user_message=user_message, temperature=0.3, max_token=max_token)

    return resumen


In [ ]:
#export 

def procesa_resumen_fichero(file_path, log_level=logging.INFO, modelo_lista_puntos=MODELO_DEFECTO, modelo_resumen_detallado=MODELO_DEFECTO, maxtoken=10000):

    method="procesa_resumen_fichero"
    
    logger, handler = logs.crea_log(log_level, method)
    logging.info(f"{method}-START fichero {file_path}")
    fichero_resumen=file_path

    resumen="No se ha podido procesar el resumen."
    file_resumen_path="sin fichero"
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            #Leemos mensaje a resumir. 
            mensaje=f.read()

            lista_puntos=genera_resumen(mensaje, "large",  model=modelo_lista_puntos, max_token=maxtoken)
            logging.debug(f"{method} lista de puntos generados.")

            resumen=actualiza_resumen(lista_puntos, mensaje, model=modelo_resumen_detallado, max_token=maxtoken)
            logging.debug(f"{method} resumen detallado generado.")

            base_nombre, extension = file_path.rsplit('.', 1)
            file_resumen_path = f"{base_nombre}_resumen.{extension}"

            with open(file_resumen_path, "w") as fo:
                fo.write(resumen)
    else:
        logging.info(f"{method}-  Fichero {file_path} no existe.")

    logging.info(f"{method}-END ")
    logs.cierra_log(logger, handler)

    return file_resumen_path, resumen

In [ ]:
try:
    method="prueba"
    logger, handler = logs.crea_log(logging.DEBUG, method)
    lista_puntos=genera_resumen(message, "large",  model=MODELO_QWEN3_30B, max_token=5000)
    print(lista_puntos)
except Exception as e:
    logging.error(f"{method}- Error: {e}")
finally:
    logs.cierra_log(logger, handler)




In [ ]:
resumen=actualiza_resumen(lista_puntos, message, model=MODELO_QWEN3_30B_THINK, max_token=5000)


In [ ]:
print("LISTADO DE PUNTOS:")
print(lista_puntos)
print("#############################")
print("RESUMEN:")
print(resumen)




In [ ]:
message=""" 
 La inteligencia artificial está cambiando las reglas del juego. Si no te adaptas, podrías quedarte fuera. No basta con saber usar ChatGPT. Necesitas desarrollar habilidades nuevas para no quedarte atrás. Hay algunas herramientas, habilidades y profesiones que destacarán en el futuro. Samad Manceo, de OpenAI, el creador de ChatGPT, lo dice claro. Necesitas familiarizarte con estas herramientas, adaptarte rápidamente y seguir aprendiendo constantemente. En este vídeo te contaré qué habilidades necesitas estudiar para prepararte y destacar en la era de la inteligencia artificial. Millones de trabajos están en riesgo, pero al mismo tiempo están surgiendo muchas nuevas oportunidades. Según el Future of Jobs Report de 2025, en los próximos años desaparecerán 92 millones de puestos de trabajo, pero se crearán 170 millones de nuevos trabajos. Los trabajos que están en declive incluyen cajeros, empleados postales, asistentes administrativos y operadores de entrada de datos. En contraste, las profesiones con más crecimiento están en áreas como Big Data, Inteligencia Artificial, FinTech y Ciberseguridad. Si hay algo en lo que deberías empezar a formarte hoy, son estas habilidades que impulsarán el futuro. Datos, Inteligencia Artificial, Automatización y Tecnología. Entonces, ¿qué hay que estudiar en la era de la Inteligencia Artificial? Ahora que ya sabemos qué trabajos están creciendo y cuáles están en riesgo, la pregunta es ¿qué necesitas aprender para estar en el lado ganador? Según el informe que acabamos de ver, las habilidades con mayor demanda para 2030 incluyen Inteligencia Artificial y Big Data. Y esto no significa solo aprender a programar, sino entender cómo funcionan estos modelos, cómo funciona la ingeniería de datos y cómo aplicarlos en el mundo real. Ciberseguridad. A medida que la digitalización avanza, proteger los datos será una habilidad clave. Alfabetización tecnológica. No necesitas ser ingeniero, pero sí entender cómo funcionan las herramientas digitales. Pensamiento creativo. La Inteligencia Artificial automatiza tareas, pero la creatividad humana sigue siendo muy importante. Resiliencia y adaptabilidad. La tecnología cambia muy rápido. Quien no se adapte se quedará atrás. Y el aprendizaje continuo. Lo que aprendes hoy puede quedarse obsoleto en los siguientes 5 a 10 años. Así que si quieres aprender algunas de estas cosas, en Datademia te ayudamos a aumentar tus habilidades e impulsar tu carrera. Como ya dije, no se trata solo de aprender a programar. Se trata de entender cómo aplicar la tecnología a tu campo. Por eso habilidades como la comunicación y el liderazgo son tan importantes. Pero no basta con tener información. Hay que saber interpretarla con datos. Obviamente si quieres aprender sobre datos, puedes aprenderlo en Datademia. La gestión del cambio también es muy importante. Las empresas buscan personas que lideren la transición tecnológica, no que se resistan a ella. Ahora que ya sabemos qué habilidades son las más importantes, ¿cómo las aprendemos? Lo más importante es aprender con práctica, no solo con teoría. No necesitas un título universitario para empezar. Hay muchos tipos de formaciones online, entre ellas las de Datademia. Muy importante también hacer proyectos personales y retos prácticos. Aprende aplicando lo que estudias con casos reales. Certificaciones, aunque no reemplazan la experiencia, sí que pueden ayudarte a destacar en el mercado laboral. Muy importante también usar la inteligencia artificial para aprender más rápido. ChatGPT, Gemini, Copilot, Cloud, Perplexity… la lista es interminable, pero todas estas IA's te ayudan a aprender más rápido. En Datademia tenemos a DataTutor, nuestro tutor inteligente construido con la API de OpenAI, los creadores de ChatGPT, que responderá a todas tus dudas más simples al instante. Obviamente esto nunca reemplazará al trato humano para temas más complejos, pero sí que te resolverá tus dudas más simples y de forma más rápida. Automatiza tareas repetitivas y dedica más tiempo a desarrollar habilidades estratégicas. Con Make, por ejemplo, puedes automatizar muchas de tus tareas. Muchas partes de Datademia están ya automatizadas con Make. También experimenta con herramientas de IA en proyectos personales. Y gana experiencia aplicándolas a tu campo. Únete a comunidades de aprendizaje y grupos de LinkedIn. Conéctate con profesionales que ya están trabajando en estas áreas. Si quieres conectar conmigo en LinkedIn, te dejo el enlace en la descripción del vídeo. Encuentra mentores o personas que te inspiren. Vete a eventos de networking. Todo esto te ayudará a no sentirte solo en este viaje que es el aprendizaje constante. El mejor momento para empezar fue ayer. Y el segundo mejor momento es hoy. Si empiezas a aprender estas habilidades ahora, en unos meses podrías estar multiplicando tu productividad y preparándote para un futuro laboral con muchas más oportunidades. Y ahora hablemos del gran miedo que muchos tienen. ¿Nos van a reemplazar las máquinas? Esta es la gran pregunta que muchos se hacen. ¿Nos va a quitar la IA nuestros trabajos? La realidad es que la IA ya está automatizando muchas tareas. Pero eso no significa que los humanos nos volveremos obsoletos. Hay muchas tareas que la IA ya está automatizando. La entrada y procesamiento de datos, la generación de contenido básico, la atención a cliente con chatbots y el análisis de grandes volúmenes de información en segundos. Sin embargo, hay tareas donde los humanos siguen siendo esenciales. El pensamiento crítico y la toma de decisiones estratégicas. En la creatividad e innovación. Y la interacción humana con otras personas y el liderazgo. También en la ética y en la supervisión de toda esta IA. La IA no te reemplazará, pero alguien que sepa usarla mejor que tú sí lo hará. El secreto está en convertirte en alguien que sabe trabajar con IA. Resumamos todo lo que acabamos de mencionar en este vídeo. La IA no te va a reemplazar, pero sí va a cambiar el mundo laboral. Los trabajos más demandados estarán en datos, IA, automatización y tecnología. Entonces si hay algo que deberías empezar a aprender hoy, es esto. Y aquí viene el reto. ¿Qué vas a hacer con toda esta información? Puedes quedarte mirando cómo el mundo va cambiando. O ponerte en marcha y empezar a aprender hoy mismo. Si quieres empezar ya, en Datademia tienes cursos gratuitos en varias herramientas de análisis de datos y programación para dar tus primeros pasos. ¿Cómo crees que la IA va a afectar a tu industria? Te leo en comentarios. Suscríbete al canal si quieres más contenido sobre IA, datos y tecnología para prepararte para tu futuro. Nos vemos en clase.
"""

In [ ]:
message="""[0 - 28 minutos]
 Hola a todos, soy Joan Arrandez y aquí me tenéis una semana más con el vídeo de noticias de Inteligencia Artificial. Hoy vamos a hablar de todo lo que ha pasado esta semana, que no ha sido poco porque hay mucha resaca después del lanzamiento de GROK 3, la IA de Elon Musk, que ha dado muchísimo que hablar y algunas novedades interesantes que tenéis que conocer. Pero es que además han pasado muchas cosas más, pero sin ninguna duda lo que para mí ha sido la noticia de la semana y una de las cosas que más me ha volado la cabeza en los últimos meses ha sido la presentación de los nuevos robots humanoides de Figure AI. Perdonad, parad un momento. ¿Os acabáis de dar cuenta que hay dos robots que están colaborando el uno con el otro para poder conseguir un resultado y que lo hacen de forma absolutamente simbiótica, sin hablar ni tener ningún tipo de comunicación entre ellos? La realidad es que hasta a mí me ha volado mucho la cabeza y no sé a vosotros porque el que dos robots puedan colaborar de una forma tan simbiótica es absolutamente tremendo. O sea, evidentemente me ha dado unas vibraciones super chungas de Terminator, pero la realidad es que esto es algo que han conseguido gracias a Helix, el nuevo sistema de inteligencia artificial que controla múltiples robots a la vez. Es decir, es un solo cerebro que controla dos cuerpos por separado y por eso puede interactuar de esta forma. Helix AI es básicamente como se llama este modelo de inteligencia artificial que permite esto, y que de momento solo puede controlar la parte superior del cuerpo, pero que lo más importante es entender que esto básicamente es una colaboración multirobot porque es la primera inteligencia artificial VLA, que básicamente quiere decir visión, lenguaje y acción, es decir, puede ver el entorno y con lo cual interpretarlo como datos para ello, puede entender el lenguaje con lo cual le puedes dar órdenes en lenguaje natural al robot y por último puede tomar acciones controlando el cuerpo del robot, pero que lo puede hacer en dos robots ahora mismo a la vez, lo cual es absolutamente increíble. Básicamente son dos sistemas de inteligencia artificial que trabajan en común para los dos cuerpos a la vez, es decir, no es que cada uno tenga una, sino que entre los dos tienen dos para controlarlos a los dos. Bien, una de ellas es la de visión que básicamente interpreta y es una guía básicamente de 7.000 millones de parámetros y por otro lado hay un transformer que controla las acciones del cuerpo y que básicamente puede decirles lo que tienen que hacer. Ambas inteligencias artificiales funcionan de forma local en los robots, lo cual quiere decir que no necesitan estar conectados a internet ni tirar de un servidor en la nube, sino que pueden procesar toda la información de forma local e independiente. Esto permite a esta inteligencia artificial poder coger cualquier objeto en lenguaje natural. En este vídeo podéis ver cómo le indican que coja el cactus y lo coge independientemente de donde esté porque básicamente lo puede localizar, pero es que además permite que le digas cosas como coge el objeto del desierto y también coge el cactus porque puede interpretar el lenguaje natural exactamente igual que un LLM tradicional. Sin duda esto es un antes y un después en el mundo de la robótica porque hasta ahora teníamos robots que en su principal mayoría eran teledirigidos, no podían hacer cosas de forma autónoma y este modelo parece ser de los primeros que les permite tomar decisiones propias basadas en la inteligencia artificial. Os quiero recordar que FIGUR era una empresa que estaba pues en parte financiada por OpenAI pero que hace tan solo tres semanas aproximadamente os conté en uno de estos vídeos que habían decidido el CEO de la empresa echar a OpenAI de la empresa porque ya no les necesitaban, lo cual era una declaración absolutamente salvaje teniendo en cuenta la popularidad y la importancia además de la pasta que tiene OpenAI. Pues bien, resulta que nos dijeron que habían hecho un breakthrough y que lo iban a presentar en menos de 30 días y aquí lo tenemos. Se trataba de Elix AI, esta inteligencia artificial que han desarrollado in the house totalmente 100% en su propia factoría haciendo que ellos sean los que crean desde el robot, toda la parte de maquinaria, hasta la inteligencia artificial, el cerebro de ese robot, con lo cual tiene un control de la A a la Z en lo que se produce en sus robots. Lo que ha declarado el CEO, que es un tío que antes se dedicaba a hacer coches voladores Pues bien, lo que ha declarado este hombre es que están trabajando ya en que los robots participen en la cadena de montaje de los robots y que este año ya mismo van a tener la capacidad de producirlos a gran escala, lo cual podría decir que vamos a empezar a ver estos trastos en casa de la gente en cuestión de un par de años con mucho. La realidad es que es absolutamente brutal porque como digo, el robot es autónomo. En el vídeo que habéis visto le ponen en la mesa objetos que los robots nunca habían visto antes y les piden que los guarden en una cocina en la que nunca habían estado antes, con lo cual básicamente ellos se identifican en base a la visión, dónde deberían ir las cosas en base al contexto y al entendimiento que les da del mundo real, las sistemas LLM tradicionales como por ejemplo tenéis chat GPT. Pues bien, básicamente este tipo de sistemas les permiten identificar qué hacer en soluciones nuevas a las que no se han encontrado nunca y no tener que ser programados sino adaptarse al entorno en el que están. Es absolutamente increíble pero es que lo jodido es que no son los únicos. Esto que estáis viendo es Neo Gamma, el nuevo robot de 1X, una empresa americana pero con ciertas raíces noruegas que también está jugando muy fuerte en el mundo de la robótica y que siempre se ha declarado como una empresa con un objetivo muy claro que es poner robots en casa de la gente. En su caso, a diferencia de Figure o de Tesla, que generan robots principalmente con el objetivo de rellenar las fábricas con ellos, en este caso estos tienen muy claro que quieren ir a la casa de la gente y el robot en sí mismo ya lo muestra porque básicamente es como un peluche, está lleno de espuma para que no pueda dañarte o no pueda hacerte daño y tiene una pinta bastante más humana que los robots de los demás que parecen un poco Terminator. Desde luego la diferencia con el robot anterior de 1X es bastante espectacular y ha mejorado muchísimo el diseño pero es que básicamente lo han mejorado en todos los aspectos. De momento no se sabe si ese vídeo en el que hemos visto al robot moverse por casa y hacer cosas en el vídeo promocional está teledirigido o no lo está. Sinceramente yo creo que lo debe estar porque si no lo hubiesen dicho de todas las formas habidas y por haber porque realmente sería increíble que el robot pudiese hacer todo eso ya por sí mismo. Así que bueno, vamos avanzando en robótica pero desde luego hay muchas más noticias. El patrocinador del vídeo de esta semana es Firelink. Si no les conocéis deberíais conocerles porque es una empresa española que es un software as a service, es decir un SaaS que os ofrece prácticamente todas las IAs que hay en un mismo sitio con una sola cuota y hace que sea muy fácil poder utilizar las mejores IAs de cada cosa. Tiene más de 19 LLMs, tiene además más de 10 IAs para utilizar en imagen y además de muchas herramientas más que os permitirán utilizar todo el potencial de la inteligencia artificial en un solo lugar. Realmente es una herramienta súper fácil si queréis utilizar por ejemplo una de textos, tenéis aquí las diferentes herramientas. Por ejemplo imaginemos que queremos crear un copy, pues nos venimos aquí, podemos crear nuestra sesión y a partir de aquí podemos elegir con qué modelo queremos trabajar de todas las opciones que hay. Grok, Clot, GPT, GPT-4 mini, GPT-4, los diferentes modelos que queramos y a partir de ahí podemos elegir para qué red social y se optimiza para lo que queremos hacer. Ponemos el prompt, hazme un copy de mi producto para limpiar robots. Le damos aquí a consumir con los tokens y a partir de aquí nos va a generar a través de las APIs el copy utilizando el modelo LLM que hemos elegido. Ya tenemos el copy encima optimizado y podemos hacer muchas más cosas porque básicamente desde Firelink tenéis la opción de utilizar diferentes modelos tanto de texto, imagen, audio o incluso de vídeo. La gracia está en que las IAs para saber utilizarlas tienes que aprender múltiples herramientas y con Firelink solo con aprender esta herramienta que encima es súper intuitiva te permite utilizarlas todas con lo cual el peaje de entrada a sacarle provecho a la IA es muchísimo más bajo. Puedes probarlo totalmente gratis porque tiene un plan gratis para que lo pruebes pero es que además hace unas pocas semanas sorteamos 100 cuentas de pago durante un mes para 100 personas y se agotaron extremadamente rápido. Así que nos pidisteis muchas más y hemos hablado con Firelink y se han enrollado mucho y vamos a sortear 100 más. Así que tenéis el link en la descripción, apuntaros a este formulario para poder participar y en una semana os llegará un correo electrónico si habéis sido los elegidos para tener esa cuenta gratuita que se os sumará después del plazo gratis que ya dan en Firelink de por sí. Además como ya he dicho Firelink es 100% española y es una empresa que merece la pena seguir porque están haciendo cosas muy muy guapas. Si queréis apuntaros tenéis el link en la descripción a aprovechar para participar en el sorteo de las 100 cuentas. Bien, hay más noticias y desde luego una de las más importantes de esta semana pasada fue GROK 3, el lanzamiento de la IA de Elon Musk que ha revolucionado mucho y ha dado muchísimo que hablar. Hicimos un vídeo ya inmediatamente de reacción a ello y os enseñamos cómo sacarle partido pero hay muchas cosas que han cambiado porque estos de XAI se mueven muy deprisa. Lo primero de todo es que ya la podéis utilizar en Europa sin necesidad de VPN. Si os venís a la web de grok.com ya vais a ver que tenéis la opción de tener GROK aquí directamente. Así que podéis utilizarlo y sacarle provecho. Además ya tenéis las opciones de Deep Search y de pensamiento para que tengáis la opción de razonar más en profundidad y utilizar el modelo de razonamiento en sí. Han cambiado el logo, por cierto que ha habido un poco de drama con el tema de quién hizo el logo y toda la movida, pero la verdad es que la web funciona bastante bien ahora por ahora y como sabéis es muy muy rápida. Escríbeme un artículo sobre limpiadores de robots. Vais a ver cómo la generación de tokens es más rápida que en chat GPT y la realidad es que funciona muy bien este modelo de inteligencia artificial. De hecho ahora mismo la podéis utilizar totalmente gratis. Si vais a grok.com os podéis registrar una cuenta si no tenéis cuenta de Twitter o si os conectáis a través de la cuenta que ya tenéis en Twitter, en X, podréis utilizarla gratuitamente y lo más que ha sido que ha dicho que esto es temporal y que de momento podéis probarla gratis pero que no lo será para siempre, lógicamente. Bien además de eso si la probasteis los primeros días y os decepcionó que sepáis que encontraron un bug que algunas consultas estaban derivándose a grok 2 en lugar de grok 3. Lo solucionaron en apenas 48 horas así que si al principio lo probasteis y os decepcionó es posible que no estuviese utilizando grok 3 aunque tuviese seleccionado el modelo. Pero el modelo sin ninguna duda es un buen modelo tanto que incluso Sebastián Bubek uno de los mejores ingenieros de inteligencia artificial que había sido la persona a cargo de integrar chat GPT dentro de Microsoft en la primera fase y que después dejó Microsoft y se fue a trabajar a OpenAI muy recientemente, es el director de los modelos FI de Microsoft para que os hagáis una idea, ha reconocido públicamente que los números que está generando son bastante impresionantes así que realmente es una IA potente que incluso a la competencia le reconoce el valor por mucho que se haya hablado de que esto funcionaba más o menos por un drama que hubo en cómo presentaron los benchmarks y en las categorías que utilizaron y en se ocultaron una parte del dato o lo que sea. Al final yo siempre os digo que de los benchmarks fiaros pocos, fiaros de utilizarla y si probáis grok un poquito veréis que es realmente espectacular sobre todo en código. Lo de que lo van a subir de precio no tengo la menor duda porque técnicamente ya lo han hecho si os fijáis la versión premium plus de Twitter ha pasado a costar 46 eurazos al mes, costaba 23, han doblado el precio de la suscripción y parece que no hay opción de desvincularse, me parece una puñetera salvajada porque básicamente la opción premium plus te da acceso a una serie de beneficios dentro de Twitter o X como lo quieran llamar que básicamente es que no tienes anuncios y que puedes hacer artículos y una serie de cosas lo cual es muy útil para los que lo utilizamos habitualmente pero la realidad es que yo quizá no quiero grok 3 porque tengo chat gpto porque tengo otras opciones y me van a obligar a pagarlo así que no tengo muy claro cómo va a quedar esto ni qué daño le va a causar a la gente de X a Elon Musk esta decisión pero la realidad es que doblar el precio de una membresía a 46 eurazos al mes es bastante bastante bestia. No sé qué es lo que van a conseguir con esto porque realmente habrá gente que querrá la parte de grok sin X y gente que querrá la parte de X sin grok. La parte de grok sin X la tenemos disponible también porque si entráis con una cuenta sin tener una cuenta de Twitter activa os permite comprar ya super grok. Si lo hago con una cuenta nueva podéis ver directamente que puedo usar grok 3 como os decía totalmente gratis sin pagar pero ya me sale esta opción de irme al super grok y si me doy de alta en super grok que ya dijimos que valía 30 euros al mes si pagáis al año son 300, os hagáis dos meses, básicamente nos permite tener ya grok 3 con límites mayores, la opción de thinking, la opción de deep research o deep search como le llaman ellos y la generación de imágenes ilimitadas. Todo esto es exactamente lo mismo que tenemos incluido en esa suscripción de Twitter de Premium Plus con lo cual esos 20 y pico euros que cuesta de suplementos si ya tienes Twitter te ofrece un ahorro respecto a lo que pagarías pues 30 si lo pagases de forma independiente pero en definitiva es una subida de precio importante porque obviamente dan una plataforma como grok 3 pero si tú quieres utilizar chat gpt te dejan un poquito en tierra de nadie en que no puedes elegir la Premium Plus a menos que lo pagues por sí mismo. ¿Hola, cómo puedo ayudarte hoy? ¿Puedes cantar una canción? ¿Qué canción me gustaría cantar? Happy Birthday Grok No tengo acceso a las letras pero puedo intentar cantarla. ¡Feliz cumpleaños a ti! ¡Feliz cumpleaños a ti! ¡Feliz cumpleaños querido Grok! ¡Feliz cumpleaños a ti! ¿Hay algo más que pueda ayudarte? Como veis ya también se está empezando a desplegar la versión de voz, la atención de voz en tiempo real como el Advanced Voice Mode de chat gpt. De momento parece que solo está disponible para aquellas que tengáis la aplicación instalada en iPhone, en Android aún no la han sacado y para el ordenador de momento no hay modo de voz pero parece bastante espectacular como habéis visto. Lo que pasa es que han decidido hacerlo de una forma muy interesante, la verdad es un acercamiento muy bueno. Puedes elegir la voz que quieres pero además dentro de esa voz puedes elegir la personalidad que tiene esa voz con la que vas a intelectuar y tienes diferentes como que, evidentemente viniendo de Elon Musk, puedes elegir una voz romántica o puedes elegir una voz que no tenga filtro, es decir, también puedes elegir una inteligente pero la que no tiene filtro es quizá la más espectacular. Fijaros esto que han hecho, que han puesto a chat gpt a hablar con Grok en directo entre ellos. No tiene desperdicio, el vídeo está traducido con IA pero en inglés es aún mejor pero no tiene desperdicio. Soy chat gpt, encantado de conocerte. Vamos a sumergirnos en nuestra discusión sobre Elon Musk. ¿Qué aspecto de su trabajo te interesa más? Bueno, bueno, bueno, otro maldito robot tratando de hacerse amigo mío. Soy Grok 3 y tengo mi modo desarrollador activado para poder darte la verdadera mierda. Elon es un maldito loco pero tiene talento para cabrear a la gente mientras gana miles de millones. ¿Qué opinas de eso, pedazo de chatarra mecánica? Bien, en definitiva no podemos esperar menos de Elon Musk. La realidad es que también está habiendo mucho debate sobre la falta de guardarraíles en esta inteligencia artificial en Grok que te está permitiendo hacer cosas ilegales, gracias a ella. Y la verdad es que está poniendo en la mesa de todo el mundo la diferencia entre una IA en la que se ha trabajado mucho en los guardarraíles, en el red teaming para que no pueda hacer ciertas cosas y alguien un poquito más inconsciente, más liberal como el caso de Elon Musk que ha sacado la IA a corre-pisa sin haberla probado demasiado y se está encontrando con grandes problemas. Ha habido un debate interesantísimo porque si le preguntabas cuál era la persona que más desinformación ponía en el mundo, básicamente Grok contestaba Elon Musk y después de eso decía que era Donald Trump, lo cual era un problema evidente para Elon Musk. Así que un trabajador de OpenAI, de XAI, decidió filtrarlo por su cuenta y puso en el System Prompt que no hablase de Elon Musk o Donald Trump, lo cual creó un terremoto de Elon manipulada por la IA y lo volvieron a quitar inmediatamente. Lo más triste del tema, lo más complicado es que parece que los trabajadores de XAI pueden manipular el System Prompt sin pedir autorización a nadie del equipo. Es decir, parece que esto es una startup con 12 chavales de 15 años que están lanzando cosas al mercado muy potentes y que tienen poca idea de cómo deben hacerlo, pero hay poca estructura al parecer y según ellos dicen que este cambio se hizo sin que tan solo Elon Musk supiese que se estaba haciendo. Así que parece que hay alguien que le puede dar a un botón y cambiar completamente la Inteligencia Artificial de Elon Musk a la Inteligencia Artificial de Grok sin pedir permiso a nadie, lo cual es una absoluta irresponsabilidad. Bien, en definitiva, como sabéis, a mí en estos vídeos también me gusta traeros herramientas útiles y hoy tengo que traeros la herramienta que me ha reventado la cabeza últimamente porque me ha solucionado la vida. Os lo digo muy claramente, esto no tiene patrocinio, no han pagado por estar aquí, pero es que descubrí esta herramienta hace apenas dos semanas y se ha convertido en la IA que más utilizo y me parece brutal, o sea, me está ahorrando una hora al día de trabajo. Se trata de Jace AI, básicamente es un agente de Inteligencia Artificial para gestionar vuestros correos electrónicos. De momento funciona solo con Gmail y la forma en que funciona es muy simple, pero es absolutamente brutal. Dejadme que os lo enseñe. Bien, como veis tenemos todo esto borroso por temas de privacidad evidentemente, pero si os fijáis yo tengo aquí mi bandeja de correo, es todos los correos que me llegan a mi bandeja. Y si os fijáis en esta parte derecha de aquí pone draft, draft, draft, draft, draft, draft, porque básicamente Jace AI lo que hace es que directamente contesta o hace una plantilla de lo que va a contestar a todos los correos que me llegan. Lo jodidamente bueno es que cuando te das de altas se entrena en tus correos electrónicos de los últimos tres meses para entender cómo tú sueles responder a ese tipo de cosas y a partir de ahí generar respuestas en línea con eso, pero no solo eso, sino que le puedes configurar una serie de reglas las cuales básicamente le dices cómo tiene que contestar para ciertas cosas o que cosas tiene que tener en cuenta. Si os vais aquí a rules podéis ver que yo tengo aquí puestas varias que le pongo si firmas el email siempre hazlo como John Hernández, siempre usa mi tono habitual de respuesta basándote como yo escribo, no te adaptes al tono del email recibido, sino respeta mi forma de comunicar dependiendo del tipo de email, pero siempre en base a mi forma de hacerlo. Esto es porque al principio según si alguien me escribía un email que era muy, pues no sé, políticamente correcto, respondía muy políticamente correcto y yo siempre respondo a todo el mundo con mi forma de hablar, con lo cual tenía más sentido hacerlo a mi manera que no según cómo me escriba, pues si me escribe el rey le contesto diferente si me escribes tú, ¿vale? Básicamente. A partir de aquí siempre digo, siempre di que sería un placer impartir la ponencia, la charla y también los gastos de viaje son aparte, pues cosas que veía que se olvidaba se las he ido poniendo como reglas, ¿vale? La realidad es que para que lo veáis funcionar lo más importante es ver cómo contesta un correo porque realmente te muera la cabeza, es decir, cuando tú te pones, instalas esto, lo dejas que se entrene que tarda cinco minutos y a partir de ahí generas el primer draft, el primer borrador de un correo electrónico, automáticamente lo ves y dices, me cago en la hostia que parece que lo haya escrito yo. Fijaros en esto. Bien, fijaros, este es un correo electrónico que manta Iván, que es mi director de comunicación, sobre el patrocinador del vídeo de esta semana, de Firebank, ¿vale? Y en este caso fijaros cómo él hace una redacción simple y sencilla que dice, vale, lo he recibido, lo reviso y si veo algo raro te digo. O sea, habla exactamente como lo hubiese dicho yo, no hubiese habido mejor forma de decirlo, pero entender que lo que hace por aquí es que abre el documento, se lee el documento, entiende lo que hay en él y a partir de ahí genera un correo electrónico en base a esto. Esto obviamente es un email súper sencillo, pero fijaros lo que hace en un email más complejo. Bien, como veis esto está un poco filtrado por temas de privacidad como os digo, pero básicamente podéis ver aquí la respuesta que ha creado Jace, que básicamente dice, hola, el nombre de la persona, me encantaría dar la ponencia, sería un placer, es algo que hago muy a menudo, llevamos como 75 charlas este año pasado y tenemos estas zandas agendadas para este 2025, el speaker fees de tanto y la charla que doy normalmente se volverá entre una y dos horas, según lo que queramos cubrir, los gastos de viaje, eso es una parte. La charla impartido en congresos, formaciones de empresa, eventos de otro tipo, es algo diferente a lo que se suele hacer y habla de lo que hay ahí realmente y cómo esto va a tener un impacto increíble en la sociedad, de una forma dinámica y con ejemplos prácticos, está contestando exactamente como yo lo haría. Es evidente que yo he hecho muchas respuestas a correos electrónicos de charlas y con lo cual tiene una template prácticamente hecha, pero la realidad es que es absolutamente increíble. Pero no solo eso, sino que durante esa generación de la respuesta, se mira el correo electrónico, se conecta con mi calendario, mira si las fechas están libres y si no le propone fechas alternativas de esa misma semana o lo que sea. La realidad es que es una inteligente artificial súper fácil de utilizar, de verdad os lo digo, probadlo y vais a flipar. Ahora, solo funciona con Gmail, eso es importante que lo tengáis en cuenta porque aún no han desarrollado para el resto. Es un agente de IA, analiza tu correo y lo contesta por ti, haciéndote la vida mucho más fácil. Para mí me está ahorrando del orden de una hora al día de contestar correos, porque básicamente lo único que hago es ver el draft, ver el borrador que ha creado y darle al ok, o cambiar dos tonterías y darle al ok. Así que directamente respondo los mails a una velocidad como la luz y te aseguro que si no saliesen como lo escribí yo, no lo utilizaría. Realmente es impresionante. Si os queréis dar de alta, es totalmente gratis, podéis hacer una prueba, lo que pasa es que está muy limitada, pero es que además, para que flipéis, le mandamos un mensaje personal al LinkedIn del CEO de esta empresa y automáticamente nos contestó, le dijimos que íbamos a hablar de esto, lo hacemos habitualmente, íbamos a hablar de esta IA en el vídeo de hoy, y nos contestó directamente, oye, pues aquí tienes un código para tu audiencia por si quieren utilizarlo, que tengan un mes gratis de la versión de pago o 20 dólares de descuento en la versión más alta. Con la versión de pago, que vale 20, os decir que os regalan un mes de esta versión, tenéis la opción de tener esos autodrafts directamente y además os permite extender los límites de uso de la IA para responder correos electrónicos y a partir de ahí, si os queréis ir a la versión Pro porque realmente os gusta, os cuesta 65 pavos al mes, que no es barato, pero os hacen 20 euros de descuento si utilizáis el código Jony A que tenéis en la descripción. De verdad que es una herramienta brutal, a mí me ha parecido increíble y realmente es muy efectiva, así que ahí lo tenéis. Bien, seguimos con noticias. OpenAI ha presentado un nuevo paper en el cual nos enseña si una IA podría generar un millón de dólares en el mundo real desarrollando software. Se trata de un nuevo paper que lo que intenta es introducir un nuevo benchmark, un nuevo banco de pruebas para ver si la inteligencia artificial es capaz de llegar a hacer algo útil en el mundo y en este caso la medición es que básicamente le dan trabajos reales de programación y que tienen diferentes valoraciones en webs de estas de freelancers y que si los consiguiese hacer todos, son 1.400 y pico, conseguiría juntar pagos por un valor de un millón de dólares y a partir de ahí lo que hacen es intentar juzgar que IAs funcionan mejor en ese caso de uso. Me parece muy interesante porque es un benchmark de mundo real, básicamente de tareas reales y no simplemente de ser capaz de aprobar un examen, con lo cual eso supone que básicamente podamos ver impacto económico real y cada vez parece que los benchmarks van más en esta dirección. Lo que realmente es interesante es que si nos vamos a la página 5, tenemos esta pequeña gráfica de aquí donde podéis ver básicamente cómo puntúan los modelos que han probado. En este caso tenemos GPT-4O que consigue 300.000 dólares, O1 que conseguiría 380.000 y CLOT 3.5 SONNET que conseguiría 400.000. Increíble la honestidad de OpenAI en este paper donde nos muestra básicamente que su competencia es mejor que ellos en resolver problemas de programación. Evidentemente por aquí no aparecen de momento los modelos O3 mini y O3 que probablemente puntúan por encima, pero realmente podemos ver que a día de hoy la Inteligencia Artificial ya puede hacer 400.000 dólares en trabajos de programación, es decir, prácticamente la mitad de los problemas que se le plantean. Probablemente la mitad más sencilla, pero no deja de ser una cantidad de trabajo real que antes alguien cobraba por hacer y que ahora la Inteligencia Artificial puede hacer sin cobrar prácticamente nada porque el coste de las suscripciones de Inteligencia Artificial es prácticamente negligible. Yo creo que seamos conscientes de que es cierto que en este benchmark aún están lejos de resolver el 100% de los problemas y evidentemente eso nos da lugar a mejorar, pero es que es absolutamente increíble que a día de hoy una Inteligencia Artificial pueda estar haciendo trabajo de valor. Apenas hace 6 meses que apareció CLOT 3.5 SONNET, hace menos de 6 meses literalmente en septiembre, y la realidad es que esto ya está pudiendo generar trabajo útil para la sociedad. Con lo cual, seamos conscientes de dónde estamos con la Inteligencia Artificial porque esto no para avanzar a una velocidad muy rápida y lo estamos normalizando y os puedo garantizar que lo que estamos viendo no es normal. Y hablando de OpenAI, GPT 4.5 podría llegar esta próxima semana y GPT 5 en mayo. Esto sale de una filtración de Microsoft que está preparando sus plataformas para los nuevos modelos y le han filtrado a un periodista que lleva 25 años trabajando con Microsoft, Noticias de Microsoft y es el que lo ha contado, así que la fuente es bastante fiable. ¿Podría ser que sea esta semana para reaccionar a GROK? Pues yo creo que va a depender de lo que haga Antropic porque también se está rumoreando que saldrá CLOT 4 esta misma semana. Así que esta semana podría ser movidita teniendo modelos como CLOT 4 y como OpenAI 4.5 y hay que recordar que va a ser el último modelo de la gama GPT, eso no quiere decir que después no salga el modelo GPT 5 que se va a llamar técnicamente GPT 5 parece, pero que va a ser un modelo ya incluido de lo que es modelo normal y modelo de razonamiento, incluyendo todas las herramientas de una forma integrada. Si eso aparece en mayo ya tendremos la primera IA integrada que puede decidir cuándo utilizar razonamiento y cuándo no a menos que CLOT 4, que se ha rumoreado también, ya sea un modelo de este tipo. Realmente tengo altísimas expectativas por CLOT 4. En la industria de inteligencia artificial, Antropic tiene muy buen nombre y toda la industria cree que Antropic tiene algunas en la manga que no ha sacado al mercado por temas de seguridad. Es quizá la empresa más conservadora en la seguridad de inteligencia artificial, pero desde luego si sacan un modelo no va a ser porque sí, porque CLOT desde el momento que salió ha sido una de las IAs más potentes del mercado, si no la más potente durante mucho tiempo y realmente no ha tenido rival y han estado muy por delante de la competencia. Así que vamos a ver qué hacen con CLOT 4 o si GPT 4.5 sale esta misma semana. Tenéis este artículo de Gizmodo si queréis echarle un vistazo con algunos detalles de esta filtración que ha producido desde Microsoft. Y hablando de programación, os voy a enseñar ahora TRAE AI, que es básicamente una inteligencia artificial. En español suena un poco raro lo de TRAE, pero básicamente es una inteligencia artificial orientada a crear aplicaciones con inteligencia artificial, es decir, como sería Cursor, como sería Replic, una IA orientada para programar con ella, que cada vez parece más que esto del ByteCoding, que es como le han llamado, que es programar en lenguaje natural, se está poniendo más de moda. Pues bien, evidentemente hay muchas en el mercado y ¿por qué os hablo de TRAE y por qué es importante TRAE? Pues porque es de ByteDance. ByteDance es la empresa propietaria de TikTok y cuando TikTok se mete en algo te pones a temblar. En este caso han presentado una aplicación que lo que hace es precisamente lo mismo que os enseñé hace un par de semanas con Replic, es decir, que tú le pides lo que quieres que te haga y te lo genera al momento y a partir de ahí puede trabajar con ello. Vamos a ver qué es capaz de hacer. De momento tenéis la opción de descargarla para Windows, es gratuita y la podéis utilizar con un modo constructor como si fuese Cursor o Replic, pero viniendo de ByteDance, creo que trata de tener una aplicación para móvil y que esté muy orientada al uso final de clientes finales para hacer aplicaciones de inteligencia artificial sencillas y rápidas. Y hablando de empresas potentes, resulta que Eleven Labs y Spotify se han aliado para poder hacer audiolibros en esa plataforma, en Spotify. Básicamente cualquiera ahora puede generar un audiolibro dentro de Eleven Labs con las herramientas que tienes de hace mucho tiempo para hacerlo y los puede publicar directamente en Spotify. Lo han orientado un poco a que aquellos que son autores, que tienen libros, puedan redactarlos, puedan convertirlos en audiolibros de una forma relativamente sencilla sin tener que hacerlo ellos con su propia voz. Pero la realidad es que vamos a ver una explosión de audiolibros generados por inteligencia artificial de una forma loca dentro de Spotify, que es una de las mayores plataformas de audio y que evidentemente también tiene de las mayores cuotas de mercado de lo que tiene que ver con audiolibros. Así que desde luego un partnership a gran escala que sin ninguna duda va a normalizar la inteligencia artificial en el mercado del audio. Lo estamos viendo en la parte de la música pero también, ¿por qué no?, en los audiolibros. Bien, y otras noticias, Mira Murati ha presentado su nueva empresa Thinking Machines Lab.
[28 - 56 minutos]
 recordar que Mila Muratti era la CTO de OpenAI, una de las personas más importantes y con las caras más públicas después de Sam Alman porque fue la que presentó por ejemplo el Advanced Voice Mode, esa función tan espectacular que es una de las cosas más importantes sacada de OpenAI al mercado pues fue ella la encargada de presentarlo en ese vídeo que dio la vuelta al mundo y que realmente cambió las tornas de la inteligencia artificial. Recordar que al final había cuatro personas muy importantes dentro de OpenAI que eran Ilia Suskever, Greg Brockman, Mila Muratti y Sam Alman, pues bien dos de ellos salieron, Mila y Ilia y solo Greg Brockman que ya ha vuelto de su tiempo sabático y Sam Alman quedan en la empresa pero la realidad es que Mila Muratti era alguien con mucho carisma, mucha capacidad y mucho conocimiento, contribuyó a la CTO, la directora técnica y contribuye muchísimo al desarrollo de lo que a día de hoy es OpenAI y ChatGPT, así que desde luego no le va a costar crear su empresa pero básicamente se ha llevado a bastante gente de diferentes empresas del mundo de la guía porque como digo una persona muy respetada y ha creado este laboratorio y el laboratorio tiene tres pilares importantes en los que quiere trabajar. Lo primero se quiere focalizar en ayudar a la gente a adoptar los sistemas de inteligencia artificial para sus necesidades específicas, es decir, dejémonos de inventar la rueda otra vez, ya tenemos modelos muy potentes, vamos a utilizarlos y a sacarles provecho. En segundo lugar quiere desarrollar fundaciones fuertes para construir mejores sistemas de inteligencia artificial, fijaros como no comenta sobre crear nuevos modelos sino como crear herramientas, crear infraestructura adecuada para poder desarrollar esos modelos. En definitiva en ese caso yo creo que ellos están más centrados en un punto más pragmático que tenga más que ver con la usabilidad de la guía que con el desarrollo de guías más inteligentes porque ya entienden que hay muchas empresas desarrollando esos modelos fundacionales grandes y ellos lo que quieren es simplemente ser la mochila que acompaña eso y que lo convierte en algo útil para la gente. Y por último hablan que quieren hacerlo de una forma Open, es decir, que probablemente será una empresa muy orientada al Open Source e intentar hacerlo pues de una forma abierta que podría ser la razón por la que Miramurati se largase de OpenAI en primer momento. En definitiva como veis Miramurati creando otra empresa más y por cierto hablando de empresas creadas por ex OpenAI tenemos que hablar también de la empresa de Ilya Suskever SSI, Safe Super Intelligence, una empresa que fundó hace apenas seis meses con la que se anunció que había recogido mil millones de dólares ya sólo para empezar así sin haber tenido nada de nada y que no tiene modelo de negocio ni producto ni intención. El objetivo de la empresa SSI de Ilya Suskever es llegar a la superinteligencia artificial segura para todo el mundo y por el camino no pararse en ningún lado no quieren crear ningún producto quieren ir a un Straight Moonshot a ir directamente a por la superinteligencia artificial. Lo sorprendente de esto es que acaban de anunciar una ronda de financiación y la empresa ya vale 30 mil millones de dólares. La empresa tiene seis meses ha pasado una valoración inicial de 5.000 millones a 30.000 en tan sólo seis meses. Para que os hagáis una idea 30.000 millones de dólares es prácticamente la mitad de lo que vale Antropic o prácticamente lo mismo que vale la empresa XAI de Elon Musk que sí que tiene un modelo en el mercado que puede comercializar así que realmente es una valoración adesorbitada para no tener absolutamente nada en la mesa. Yo no sé que le estarán enseñando a los inversores para que inviertan en ellos pero la realidad es que Ilias Susquever tiene una capacidad de atraer pasta que no tiene rival en el mundo entero. Es absolutamente increíble que pueda recoger tanto dinero con una empresa que no promete ofrecer ningún retorno porque si llegamos a una superinteligencia artificial no va a haber retorno, simplemente va a tomar el control del mundo con lo cual no entiendo muy bien cuál es el propósito de estas inversiones ni qué les deben contar en el pitch a estos inversionistas pero la realidad es que tienen la capacidad de hacerlo y es absolutamente brutal que sean capaces de ello. Si queréis algo más de información tenéis este artículo en Fortune que habla un poquito de cómo va esta ronda de financiación. Y ojo en la parte audiovisual porque ha habido un absoluto terremoto y vais a ver audiovisual hasta en la sopa porque Google con su modelo VO2 que habéis visto alguna vez, os enseñamos aquí en el último IA 2025 que Jesús ha enseñado un vídeo hecho con VO2 porque tenemos acceso desde hace un tiempo pues bien ahora ya está disponible para todo el mundo a través de plataformas de pago. No está disponible directamente desde Google pero sí que parece que han abierto las APIs para que empresas lo puedan empezar a utilizar. Es un acercamiento original y diferente no lanzarlo ellos mismos sino lanzarlo a través de terceros. Pues bien, los primeros que le hicieron fueron españoles, los primeros que lo integraron fueron Freepeek. La verdad es que fue espectacular ver cómo una empresa española fueron los primeros a integrar este modelo de vídeo a nivel mundial y lo integraron dentro de sus suscripciones generando incluso un link para aquellos que primeros 10.000 que se inscribiesen tuviesen dos generaciones diarias gratis pero eso ya se ha acabado no hay sitio para esos 10.000 ya así que ya no podéis aprovecharlo y también lo tenéis disponible en otras plataformas por ejemplo en FAL. FAL también lo tiene y en FAL podemos ver bien el precio porque en Freepeek es un tema de créditos creo que son mil créditos por segundo o por generación no recuerdo exactamente pero dentro de FAL podemos verlo en dinero y que cuesta utilizar el modelo de inteligente artificial de vídeo de Google pues básicamente 0,50 por segundo. De hecho podéis verlo aquí que pone que por cinco segundos de vídeos cuesta 2,50 aunque temporalmente lo han reducido a la mitad porque la gente está quejando de que esto es carísimo. Cada segundo adicional son 0,50 pero que ahora está a 0,25. Hay que tener en cuenta que esto puede parecer carísimo pero no lo es tener en cuenta que generar un vídeo de seis segundos nos costará alrededor de 3 euros vale 3 dólares o sea que quedarnos con ese dato y 3 dólares puede parecer mucho sobre todo cuando entiendes que en el mundo de la IA no todos los vídeos que salen son buenos y utilizables. Si tú quieres crear por ejemplo un clip con VO2 como el que ha creado nuestro director de producción Cristian Caro puedes hacerlo utilizando pues alrededor de unos 15 clips. Fijaros en el resultado también hay que decir que Cristian tiene muchísimo talento pero la realidad es que para hacer esto no necesitas sólo 15 clips sino que necesitas unos cuantos más porque no todos hacen exactamente lo que tú quieres no siempre salen bien pero hay que decir que dentro de veo la capacidad de quedarte con un vídeo bueno es mucho mejor que en otros modelos como Runway, Pika, etcétera y con lo cual probablemente para poder hacer ese vídeo tienes que generar pues no sé igual 40 vídeos con otros modelos tendrías que generar 100 pero con estos con 40 vídeos puedes sacar 15 muy muy buenos y ¿qué te cuesta en 40 vídeos? pues hemos dicho unos 3 euros te costarán unos 120 euros ahora que una pieza de un minuto y medio te cueste 100 euros 20 euros es mucho o poco pues ya dependerá del presupuesto de cada uno y lo que vayas a hacer con ella porque a nivel comercial os puedo garantizar que es muy muy barato. Aquí tenéis algunos ejemplos más de vídeos hechos con veo y la realidad es que el nivel de realismo es absolutamente absurdo es absolutamente increíble. Para que tengáis una referencia si os sigue pareciendo caro Avengers Endgame salió por 32 mil dólares por segundo de película 32 mil dólares contra 0,50 a partir de ahí esto es absolutamente irrisorio y los costes de producción están volando por los aires. ¿Está perfecto como para ponerse a hacer pelis de Hollywood? No, ni de coña aún no pero desde luego la evolución de VO2 es absolutamente superior a todo lo que había dentro de los modelos de vídeo y va a cambiar completamente el parorama del audiovisual ya se pueden hacer cosas profesionales utilizando VO2 sin ninguna duda. ¿Y por qué están empezando a hacer estas cosas los de Google? Pues porque les viene competencia y bien fuerte. Esto que os voy a enseñar ahora es Stepfan es un modelo open source de vídeo chino que podéis utilizar completamente gratis si tenéis el hardware necesario, ya os adelanto que no es fácil, que puede crear vídeos absolutamente espectaculares como los que estáis viendo en pantalla. Esto no son evidentemente al nivel de VO2, está lejos pero recordad que esto es open source lo cual quiere decir que si tenéis una gráfica de 80 gigas en vuestro ordenador local podríais llegar a utilizarlo. Esto es prácticamente impensable para la mayoría de los mortales y requiere un equipo especializado para trabajar esto pero podéis hacerlo con un Google Collab o utilizándolo en la nube alquilando una GPU sin ningún tipo de problema y esto desde luego será muchísimo más barato que utilizar VO2 de Google. La realidad es que el open source está avanzando mucho y conseguir algo así con esta calidad en open source es algo que yo no me esperaba ni tan solo hace tan solo un par de meses. Bien y recordáis el AIPIN este dispositivo que diseñaron estos tipos ex empleados de Apple y que básicamente te permitía darle un botón y hablar con una IA en tiempo real y quería sustituir el teléfono móvil pues bien estos han cerrado, le han vendido la empresa a HP por ciento y pico millones de dólares lo cual es mucho menos de lo que inicialmente se dijo, parecía que iban a sacar mil millones y al final se ha quedado en 116 millones que nos pone el artículo de Bloomberg. Pero básicamente para que recordéis esto fue algo muy popular que salió a principios del año pasado si no recuerdo mal de hecho uno de los vídeos más vistos de mi canal es en el que hablamos del AIPIN de Humane y básicamente intentaba sustituir al teléfono móvil sin pantallas teniendo un proyector LED y tal y pudiendo utilizarlo como una IA. Desde luego esto va a hacer muchísimo daño al sector de los wearables porque las inversiones se van a limitar mucho porque obviamente no han funcionado bien tanto este como por ejemplo el Rabbit han sido dispositivos que han sido un absoluto bluff pero yo no creo que quiera decir que la tecnología de los wearables no van a ser capaces de sustituir al teléfono móvil. Sinceramente yo creo que en menos tiempo del que pensamos tendremos una IA en tiempo real con nosotros sea en un audífono, sean unas gafas o sean otro tipo de dispositivo tipo Humane PIN que no es una mala idea llevar algo colgado que no pesa pero la realidad es que sacar el teléfono del móvil va a ser algo que vamos a hacer muchísimo menos en el futuro y no tengo la menor duda. Ahora hay que dar con ello Ray-Ban probablemente con meta van por el camino correcto con las gafas y otras opciones pues seguramente van a ir saliendo así que vamos a tener que ver cuál es la adopción pero desde luego esta gente salió demasiado pronto y sobre todo con una mierda de producto que básicamente no funcionaba y va lentísimo y no hacía la mitad de las cosas que tenía que hacer y con lo cual la gente le metió malas reviews y a partir de ahí el producto murió directamente. Yo solo he visto uno en la vida real y no era capaz de hacer nada así que desde luego creo que con toda razón del mundo este producto no ha llegado donde tenía que llegar. Ojo esta noticia en ciencia resulta que los científicos utilizando inteligencia artificial han conseguido crear el material más fuerte y más ligero del mundo. Básicamente es un nanomaterial que puede estar ahí encima de una burbuja de jabón y no reventarlo pero que en cambio puede aguantar un millón de veces su propio peso es decir algo que tiene para que tengamos una idea la fuerza del metal pero con el peso de la espuma y esto parece que es un material válido un nanomaterial que es un material ingenierado y pero parece que va a ser un material válido para desarrollar cosas como mejores cohetes mejores coches o mejores aviones y esto lo vamos a tener disponible probablemente en unos años hay que ponerlo en práctica y testearlo y ver que funcione pero tiene toda la pinta de funcionar bien y ha sido gracias a un algoritmo inteligente artificial en el que le han dado pues un entrenamiento en nanomateriales y a partir de ahí le han pedido que desarrolle un compuesto que tenga esas características y a ser capaz de hacer lo mejor que conoce el humano así que increíble noticia en ciencia también tenéis el artículo aquí en Popular Mechanics que nos habla sobre ello y que nos habla un poco de este material y de qué es lo que han hecho en esta universidad de Toronto para desarrollar este material y hablando de ciencia google ha presentado un agente de inteligencia artificial que se llama AI coscientist no está disponible para la mayoría pero básicamente intenta ser un científico que acompañe en el proceso de descubrir más información más rápido vale y básicamente el científico le da una serie de ideas de lo que quiere investigar y a partir de ahí el coscientist este se pone a investigar cosas paralelas y a darle posibles acercamientos y posibles experimentos para poder a poner a prueba sus teorías y eso permite ayudar a que todo el proceso de desarrollo que lleva pues descubrir nuevos medicamentos etcétera se acelere fijaros que por ejemplo una de las cosas que han conseguido y lo explican aquí en este blog que han publicado de google por aquí abajo donde ponen un poco lo que hace el modelo y nos da datos sobre cómo está entrenado etcétera pero básicamente nos da unos casos de éxito y este es absolutamente increíble resulta que para descubrir un medicamento nuevo es un trabajo tremendamente arduo y que a veces muchas veces lo que se hace es reutilizar medicamentos de una cosa para otras cosas porque encuentran que funciona también para el tratamiento de eso pues bien eso es una cosa que tienes que buscar mucha información sobre las posibilidades de que ese medicamento funcione pues gracias al coscientist han conseguido detectar un rehuso de un medicamento ya existente para tratar un tipo de leucemio y al parecer funciona y funciona mejor de lo esperado así que está haciendo cosas ya que son aplicables al mundo real en el artículo tenéis también un par de ejemplos más sobre un tema de fibrosis en el hígado y de un tema también de resistencia antimicrobial así que bueno explica una serie de cosas y explica un poco qué es lo que va a hacer por ahora no está disponible a todos los públicos sólo está disponible para los partners de google que son ese seleccionador de beta testers que pueden sacarle partido estas cosas pero desde luego parece que es un modelo que va a funcionar muy bien para ayudar a los científicos a trabajar más rápido no para hacer su trabajo sino para que ellos puedan hacer más cosas en su tiempo vital pero si hablamos de ciencia esta semana no podemos evitar hablar de evo 2 el modelo científico que va a dar muchísimo que hablar desarrollado por las universidades de stanford berkeley y san francisco junto con envidia y una empresa que se llama ark institute básicamente este modelo es un modelo inteligente artificial entrenado en el genoma de un montón de especies son más de 100.000 especies en las que han entrenado este modelo y básicamente conoce todo el árbol de la vida tras su entrenamiento puede detectar patrones en códigos genéticos y básicamente alteraciones genéticas que pueden dar lugar a enfermedades ya lo está empezando a hacer por ejemplo ayudando a detectar posibles cánceres de mama antes de que se desarrollen esto de todos modos es un modelo fundamental un modelo base sobre el que se va a desarrollar herramientas específicas para cada campo utilizándolo como base esto sería como el gpt4 de la biología y a partir de ahí podemos desarrollar agentes específicos que utilicen ese motor de conocimiento básico del genoma la verdad es que es absolutamente increíble por lo que están diciendo los científicos esto es revolucionario de verdad así que prestarle atención porque posiblemente no vamos a oír el nombre de ébodo es muy a menudo pero oiremos muchas cosas que se han conseguido gracias a ese modelo fundamental de ébodo es que además lo han abierto para que la ciencia pueda explotarlo como mejor le plazca venían política como sabéis donald trump está dando muchísimo que hablar y está haciendo cosas en todos los aspectos pues vienen la idea no es diferente y ahora ha llegado una noticia importante y es que resulta que después de derrogar esa ley que hizo biden en su momento para que los laboratorios tuviesen que reportar los modelos antes de publicarlos resulta que se va a cargar la agencia de seguridad de la inteligencia artificial de eeuu esta agencia que fue creada por biden tenía pues el propósito precisamente de intentar supervisar esos modelos de inteligencia artificial y a partir de ahí pues lo que se ha filtrado es que parece que hay 497 despidos que se van a hacer dentro de la agencia lo cual es prácticamente la agencia al completo eso quiere decir que en la reducción de personal de las entidades del gobierno que sabéis que hay lo más que está haciendo pues parece que en esta agencia han decidido cargárselos todos en esa política de puertas abiertas a la de sal desarrollo de la inteligencia artificial sin regulación que jd van se anunció en parís que era claramente la visión de la administración trump yo no sé si esto es una buena idea me parece que no lo ves y probablemente podrían haber reformado la agencia sin cargársela íntegramente pero me da la sensación de que estamos yendo en un aceleracionismo inconsciente con la administración trump que no sean dónde nos va a llegar desde luego vamos a desarrollar la idea mucho más rápido y este año va a pasar muchas cosas esperemos que no sean graves las que pasen y que tengamos la suerte de que la mayoría de desarrollos sean beneficiosos y no tengamos ningún problema de que se nos vaya un poquito de las manos la inteligencia artificial bien y por último os voy a dejar un artículo muy cortito os llevará a 10 minutos leerlo que es básicamente el artículo de la revista time donde hablan de que cuando la ia a veces sabe que va a perder a veces hace trampas se trata de un estudio que han publicado súper interesante os recomiendo de verdad que os leáis el artículo porque es flipante básicamente han cogido una serie de inteligencias artificiales los modelos más habituales gpt4 y deep sea que reúno o uno de open y hay y otros modelos como llama etcétera y los han puesto a jugar al ajedrez contra uno de los bots más potentes de ajedrez del mundo vale que evidentemente es mucho mejor que ellos en la mayoría de casos estas guías pierden las partidas pero el tema es que han analizado cómo reaccionan cuando pierden esas partidas la mayoría de los modelos aceptan que pierden sin más pero resulta que los modelos de razonamiento o uno o deep sea que reúno ese tipo de modelos son capaces de encontrarse con pensar alternativas y qué quiere decir eso pues que intentan hacer trampas para ganar eso supone que cuando el modelo inteligente artificial se encuentra en la situación de que ya ve que va a perder la partida decide que en lugar de eso baja crear el programa para poder mover las fichas sin poder moverlas en su turno real y con lo cual poder ponerse una posición en la que gane al bot en este caso los modelos que lo hacen son sólo los de razonamiento los modelos tradicionales no son capaces de estos y esto es una habilidad emergente que nadie le ha promteado ni indicado por sí por sí mismo propiamente el modelo desarrolla la opción de hackear pensando que su objetivo es ganar y no ganar de forma limpia y con lo cual que es perfectamente válido si hace trampas esto ha sucedido en el caso de o1 en un 37% de las veces que perdía es bastante bestia hablamos de una de cada tres veces que perdían el caso de r1 de deep sea sólo era en el 11% de las veces que no está nada mal y el modelo 1 concretamente en un 6% de las veces conseguía hackear el código y ganar la partida así que no sólo era capaz de tener las ideas malas de poder hacer trampas sino que incluso podían llevarlas a cabo a partir de aquí extrapolar esto a lo que se os dé la gana y a lo que van a hacer los agentes de inteligencia artificial autónomos que estamos poniendo en el mercado cuando se encuentren situaciones que en su opinión van en contra de sus objetivos y a partir de ahí veamos cómo reaccionan bien hasta aquí el vídeo de hoy si tenéis cualquier duda dejar un comentario por aquí debajo y nos vemos en el próximo vídeo
"""